# Sensor Monitor

Run this notebook with `voila --theme=dark --Voila.ip=0.0.0.0 --no-browser bokeh_render.ipynb`.

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path
from itertools import cycle
from copy import deepcopy

import bokeh
from bokeh.themes import Theme
from bokeh.themes._carbon import json
from bokeh.plotting import figure, show, curdoc
from bokeh.layouts import column, row
from bokeh.models import LinearAxis, Range1d, ColumnDataSource, HoverTool

bokeh.io.output_notebook(hide_banner=True)

json=deepcopy(json)
json["attrs"]["figure"]["background_fill_color"]="#00000000"
json["attrs"]["figure"]["border_fill_color"]="#00000000"

curdoc().theme = Theme(json=json)

db = sorted(Path("..").glob("*.sqlite"))[-1]  # most recent database

DAYS_S = 60 * 60 * 24  # how many seconds in a day
NUM_DAYS = 3  # how many days of data to read
FREQ = "5min"  # frequency to group data items by

COLORS = [
    "#636EFA",
    "#EF553B",
    "#00CC96",
    "#AB63FA",
    "#FFA15A",
    "#19D3F3",
    "#FF6692",
    "#B6E880",
    "#FF97FF",
    "#FECB52",
]

SELECTED_COLS = ["temperature", "humidity", "pressure", "iaq"]
GAS_COLS = ["gas_resistance", "oxidising", "reducing", "nh3"]
RGB_COLS = ["r", "g", "b", "c"]

In [ ]:
with sqlite3.connect(db) as con:
    df = pd.read_sql_query(f"SELECT * from readings order by date desc limit {DAYS_S*NUM_DAYS}", con)

df.date = pd.to_datetime(df.date, format="mixed", errors="coerce")
df.set_index("date", inplace=True)
df.sort_index(inplace=True)
df_sel = df.groupby(pd.Grouper(freq=FREQ))  # .mean().dropna(how="all")

In [ ]:
figs = []

for idx, colname in enumerate(SELECTED_COLS):
    color = COLORS[idx % len(COLORS)]
    y = df_sel[colname].mean().dropna(how="all")
    ymin = df_sel[colname].min().dropna(how="all")
    ymax = df_sel[colname].max().dropna(how="all")
    kwargs = {"x_range": figs[0].x_range} if idx > 0 else {}

    p = figure(
        width=1400,
        height=150 + (40 if idx == 0 else 0),
        x_axis_type="datetime",
        x_axis_location="above",
        title=f"{colname.title()} = {y.iloc[-1]:.2f}",
        tools="reset,xpan,xbox_zoom",
        **kwargs,
    )

    p.line(y.index, y, color=color)
    p.varea(y.index, y1=ymin, y2=ymax, fill_color=color, fill_alpha=0.25)

    # p.background_fill_color = "#303030"
    # p.border_fill_color = "#202020"
    p.border_fill_alpha = 0
    p.xaxis.visible = idx == 0
    # p.xgrid.grid_line_color = "#f0f0f0"
    # p.ygrid.grid_line_color = "#f0f0f0"
    p.toolbar.logo = None

    figs.append(p)

y = df_sel[GAS_COLS].mean().dropna(how="all")

gasfig = figure(
    width=1400,
    height=200,
    x_axis_type="datetime",
    x_axis_location="above",
    title="Gasses",
    tools="reset,xpan,xbox_zoom",
    x_range=figs[0].x_range,
)
gasfig.yaxis.visible = False
gasfig.xaxis.visible = False
gasfig.toolbar.logo = None

for idx, g in enumerate(GAS_COLS):
    color = COLORS[(idx + len(figs)) % len(COLORS)]
    gasfig.extra_y_ranges[g] = Range1d(y[g].min(), y[g].max())
    ax = LinearAxis(y_range_name=g, axis_label=g.replace("_", " ").title())
    ax.axis_label_text_color = color
    gasfig.add_layout(ax, "left")
    gasfig.line(y.index, y[g], y_range_name=g, color=color)


y = df_sel[RGB_COLS].mean().dropna(how="all")

rgbfig = figure(
    width=1400,
    height=200,
    x_axis_type="datetime",
    x_axis_location="above",
    title="RGBL Values",
    tools="reset,xpan,xbox_zoom",
    x_range=figs[0].x_range,
)
rgbfig.xaxis.visible = False
rgbfig.toolbar.logo = None

for c, color in zip(RGB_COLS, ["red", "green", "blue", "white"]):
    rgbfig.line(y.index, y[c], color=color, line_width=2)
    rgbfig.varea(y.index, y1=y[c] * 0, y2=y[c], fill_color=color, fill_alpha=0.25)


show(column(*figs, row(gasfig), rgbfig))